# Sequence-matched quantitative simulation with an unsuppressed water reference

Run this notebook in the repository Pixi kernel. It uses the supported production `run_headless` API, the packaged frozen PRESS TE30 bundle and no anatomical dataset, Octave regeneration or fitting. The legacy demonstration remains in `headless_simulation.ipynb`.

Metabolites are specified in **mM per total tissue volume**, water in **mol/L per total tissue volume**. Receiver gain is 1. The unsuppressed reference comes from tissue water content and the shared sequence response, never from the legacy residual-water model. Calibration supports relative sequence response; it is not scanner-level absolute calibration.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fsl_mrs.utils import mrs_io

repo = Path.cwd()
if not (repo / 'pyproject.toml').is_file():
    repo = repo.parent
assert (repo / 'examples/configs/quantitative/pure_WM.json').is_file(), 'Start from the repository root or notebooks/'
os.chdir(repo)
from simulation.headless import run_headless
from simulation.calibration_resources import default_calibration_bundle

output_root = Path('outputs/notebook_quantitative_demo') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output_root.mkdir(parents=True, exist_ok=False)
print('Frozen calibration:', default_calibration_bundle().relative_to(repo))
print('Fresh output root:', output_root)


In [ ]:
def load_demo_config(name):
    config = json.loads(Path(f'examples/configs/quantitative/{name}.json').read_text())
    config['output_dir'] = str(output_root / name)
    return config

def show_pair(pair, name):
    truth, cfg = pair['ground_truth'], pair['simulation_config']
    nominal = truth['voxel_average_metabolite_concentrations_mM']
    print('Calibration:', truth['calibration_identity'])
    print('Tissue fractions:', truth['tissue_fractions'])
    print('Effective voxel water (mol/L):', truth['effective_voxel_water_concentration_mol_l'])
    display(pd.DataFrame({'voxel-average mM': nominal,
                          'WM/GM-only mM': truth['tissue_only_metabolite_concentrations_mM'],
                          'relaxation-weighted mM': truth['relaxation_weighted_metabolite_concentrations_mM']}))
    # FID convention is exp(+i*2*pi*f*t); ppm = 4.68 + f / MHz.
    ppm = cfg['frequency_origin_ppm'] + np.fft.fftshift(np.fft.fftfreq(len(pair['time_s']), cfg['dwell_time_s'])) / cfg['spectrometer_frequency_mhz']
    fig, ax = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
    for a, key, title in [(ax[0,0], 'metabolite_fid', 'Water-suppressed metabolites'),
                           (ax[0,1], 'water_reference_fid', 'Unsuppressed water reference')]:
        spectrum = np.fft.fftshift(np.fft.fft(pair[key]))
        a.plot(ppm, spectrum.real, label='Real')
        a.plot(ppm, spectrum.imag, label='Imaginary', alpha=.6)
        a.set(xlim=(5.2, .2), xlabel='Chemical shift (ppm)', ylabel='Relative signal units', title=title)
        a.legend()
    ax[1,0].bar(truth['tissue_fractions'].keys(), truth['tissue_fractions'].values())
    ax[1,0].set(ylabel='Volume fraction', ylim=(0, 1), title='Tissue composition')
    ax[1,1].bar(nominal.keys(), nominal.values())
    ax[1,1].set(ylabel='Nominal voxel-average concentration (mM)', title='Ground truth')
    fig.savefig(output_root / f'{name}.png', dpi=120)
    plt.show()
    # Full complex equality, not spectral magnitude alone.
    for label, key in [('metabolites', 'metabolite_fid'), ('water_reference', 'water_reference_fid')]:
        reader = mrs_io.read_FID(pair['saved_paths'][label])
        np.testing.assert_array_equal(reader.mrs().FID, pair[key])
        assert reader.dwelltime == .0005
        assert reader.hdr_ext['SpectrometerFrequency'] == [127.731]
    saved_truth = json.loads((Path(pair['saved_path']) / 'ground_truth.json').read_text())
    assert saved_truth == truth
    assert cfg['synthetic_calibration']['normalization']['scalar'] == 8857.353925964408
    print('Paired acquisitions and JSON sidecars:', pair['saved_path'])
    print('Maximum complex export/reload error: 0')


### Pure white matter

NAA 10, Cr 8, GPC 2, PCh 1 and Glu 8 mM; water 39.0165 mol/L. No additional relaxation, broadening, B0 or noise. Water is much larger because it is present at tens of mol/L while metabolites are at thousandths of mol/L, with proton responses already encoded. Plots use their actual relative units on separate axes; no signal is renormalized.


In [ ]:
pure_config = load_demo_config('pure_WM')
print(json.dumps(pure_config, indent=2))
pure_pair = run_headless(pure_config)[0]
show_pair(pure_pair, 'pure_WM')


### Mixed WM/GM/CSF

20% WM, 30% GM and 50% CSF. Voxel-average concentrations include the entire represented volume. WM/GM-only concentrations divide the WM/GM contributions by their combined fraction and exclude CSF. These definitions are never interchangeable.


In [ ]:
mixed_config = load_demo_config('mixed_tissue')
print(json.dumps(mixed_config, indent=2))
mixed_pair = run_headless(mixed_config)[0]
show_pair(mixed_pair, 'mixed_tissue')
assert mixed_pair['ground_truth']['tissue_fractions'] == {'WM': .2, 'GM': .3, 'CSF': .5}


### Tissue relaxation, independent linewidths, B0 and reproducible acquisition noise

TE=0.030 s, TR=2 s with explicit tissue/metabolite T1/T2. Pre-acquisition attenuation is `(1-exp(-TR/T1))*exp(-TE/T2)`; it does not replace the frozen readout decay. Additional Lorentzian widths are 2 Hz for metabolites and 7 Hz for water, without forcing equal measured peak widths. Spatial B0 offsets are -3, +7 and +1 Hz, with positive offsets defined by `exp(+i*2*pi*df*t)`. Receiver noise is added after spatial integration using seed 910 and independent metabolite/water streams.

Simulator SNR is the clean NAA acetyl spectral peak divided by frequency-domain noise SD for one real/imaginary component (orthonormal FFT); it is not a fitting-package SNR. Target metabolite SNR=50, water SNR=1000. Clean acquisitions are retained. The T1/T2 values are reproducible example assumptions, not universal tissue constants.


In [ ]:
realistic_config = load_demo_config('noisy')
print(json.dumps(realistic_config, indent=2))
realistic_pair = run_headless(realistic_config)[0]
show_pair(realistic_pair, 'relaxation_B0_noise')
repeat_config = deepcopy(realistic_config)
repeat_config.pop('output_dir')
repeat_pair = run_headless(repeat_config)[0]
for key in ['metabolite_fid', 'water_reference_fid', 'clean_metabolite_fid', 'clean_water_reference_fid']:
    np.testing.assert_array_equal(realistic_pair[key], repeat_pair[key])
print('Fixed-seed repeat: exact complex equality; clean and noisy acquisitions remain separate.')


All generated acquisitions, plots and execution records belong under the ignored output root printed above. The notebook stores no new embedded figures or execution outputs. See `docs/quantitative_simulator_user_guide.md` for CLI use, units and limitations. No fitting is performed here.
